# Status report

By Ben Welsh

Generates basic statistics from [News Homepages database extracts](https://palewi.re/docs/news-homepages/extracts.html).

In [1]:
import os
import json
import pandas as pd
import altair as alt
from pathlib import Path
from datetime import datetime, timedelta

In [2]:
this_dir = Path("__file__").parent.absolute()

In [3]:
sources_dir = this_dir.parent / "newshomepages" / "sources"

In [4]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [5]:
df = pd.read_csv(
    extracts_dir / "screenshot-files.csv",
    parse_dates=["mtime"],
    usecols=["identifier", "handle", "file_name", "mtime"]
)

In [6]:
df['date'] = df.mtime.dt.date

In [7]:
df["date"] = pd.to_datetime(df["date"])

How many total sites?

In [8]:
total_sites = len(df.handle.unique())

In [9]:
total_sites

221

How many total screenshots?

In [10]:
total_screenshots = len(df)

In [11]:
total_screenshots

23318

When did we start?

In [12]:
start_date = min(df.date)

In [13]:
start_date

Timestamp('2022-03-22 00:00:00')

How many screenshots in the last week?

In [14]:
today = datetime.now().date()

In [15]:
today

datetime.date(2022, 7, 2)

In [16]:
one_week_ago = today - timedelta(days=7)

In [17]:
one_week_ago

datetime.date(2022, 6, 25)

In [18]:
df_this_week = df[df.date > pd.to_datetime(one_week_ago)]

In [19]:
screenshots_this_week = len(df_this_week)

In [20]:
screenshots_this_week

5076

Write out data points

In [21]:
output = dict(
    total_sites=total_sites,
    total_screenshots=total_screenshots,
    screenshots_this_week=screenshots_this_week,
)

In [22]:
json.dump(output, open(this_dir / 'status-report.json', 'w'), indent=2)

Chart the number of sites by date

In [23]:
sites_by_date = df[['date', 'handle']].drop_duplicates().groupby("date").size().rename("sites").reset_index().sort_values("date")

In [24]:
sites_by_date['rolling_mean'] = sites_by_date.sites.rolling(7).mean()

In [33]:
chart = alt.Chart(
    sites_by_date.head(len(sites_by_date) - 1),
    title="Sites archived by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("sites:Q", title="Sites"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).encode(
    x='date:T',
    y='rolling_mean:Q'
)

label = chart.encode(
    x=alt.X('max(date):T'),
    y=alt.Y('rolling_mean:Q', aggregate=alt.ArgmaxDef(argmax='date')),
    text='rolling_mean'
)

# Create a text label
text = label.mark_text(align='left', dx=4)

# Create a circle annotation
circle = label.mark_circle(size=75, color="#727272")


(bars + line + circle)

alt.LayerChart(...)

In [30]:
if os.getenv("CI"):
    (bars + line + circle).save(this_dir / 'sites-by-date.png')

Chart the number of screenshots by date

In [26]:
screenshots_by_date = df.groupby("date").size().rename("screenshots").reset_index().sort_values("date")

In [27]:
screenshots_by_date['rolling_mean'] = screenshots_by_date.screenshots.rolling(7).mean()

In [34]:
chart = alt.Chart(
    screenshots_by_date.head(len(screenshots_by_date) - 1),
    title="Screenshots saved by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("screenshots:Q", title="Screenshots"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).encode(
    x='date:T',
    y='rolling_mean:Q'
)

label = chart.encode(
    x=alt.X('max(date):T'),
    y=alt.Y('rolling_mean:Q', aggregate=alt.ArgmaxDef(argmax='date')),
    text='rolling_mean'
)

# Create a text label
text = label.mark_text(align='left', dx=4)

# Create a circle annotation
circle = label.mark_circle(size=75, color="#727272")

(bars + line + circle)

alt.LayerChart(...)

In [32]:
if os.getenv("CI"):
    (bars + line + circle).save(this_dir / 'screenshots-by-date.png')